# Ripartizione di una fattura tra N persone

Divide una fattura (onorari + spese generali + cassa, opzionalmente IVA) in **N fatture** in modo che, sommandole, **ogni voce coincida al centesimo** con la fattura originale.

**Come si usa:** modifica i parametri nella prima cella di codice, poi esegui tutte le celle in ordine (*Run All*). Il risultato è nella terza cella di codice. Come esempio, il notebook è impostato su onorari di 1.844,50 € divisi tra 6 persone.

## Contesto: come si compone la fattura

Il calcolatore online [Calcolo Fattura per Avvocati e Studi Legali](https://www.avvocatoandreani.it/servizi/calcolo_fattura_studio_legale.php) costruisce la fattura partendo da *onorari*, *spese esenti* e *spese imponibili*. 

Dalla pagina risulta che:
- le **spese generali** (art. 13, comma 10, L. 247/2012; misura stabilita dall'art. 2 del D.M. 55/2014) sono opzionali, con percentuale di default del **15%** e modificabile;
- la quota per la **Cassa Forense (4%)** è calcolata automaticamente;
- l'**IVA al 22%** è calcolata automaticamente, tranne nei regimi semplificati/forfettari, dove non si applica;
- sono previsti anche spese esenti (art. 15 DPR 633/72), ritenuta d'acconto, split payment e imposta di bollo (2 € per le fatture in regime semplificato di importo superiore a 77,47 €).

> **Attenzione.** Il testo della pagina non spiega né su quale base si calcoli esattamente il 4% né quale regola di arrotondamento si usi. Il notebook assume che la cassa si applichi a *onorari + spese generali* e che ogni voce sia arrotondata al centesimo. È l'impostazione più comune, ma conviene confrontare qualche importo con il calcolatore del sito prima di fidarsi del modello.

### Il modello

Dato l'onorario `q0`:

| voce | significato | formula |
|---|---|---|
| `q0` | onorari | importo dato |
| `q1` | spese generali | 15% × `q0` |
| `q2` | cassa forense | 4% × (`q0` + `q1`) |
| `q3` | totale | `q0` + `q1` + `q2` (IVA esclusa) |

Ogni voce viene arrotondata al centesimo prima di essere usata nella successiva. L'IVA (22% su `q3`, arrotondata fattura per fattura) si può includere con `CON_IVA = True`; in quel caso `qta` contiene anche `IVA` e `Totale + IVA`.

### Il problema

Gli arrotondamenti non sono lineari: `round(a) + round(b) ≠ round(a + b)`. Dividere la fattura in N quote uguali (`splitted * N`) spesso non restituisce le stesse voci dell'originale, anche solo di un centesimo. Il notebook cerca allora importi per le N fatture la cui somma coincide con l'originale su **ogni** voce, in questo ordine di preferenza: (1) quote tutte uguali; (2) **due soli importi** (N1 persone pagano `cost1`, le altre N2 pagano `cost2`), scegliendo la coppia più vicina alla quota ideale; (3) solo se non esiste, importi liberi (anche tutti diversi).

## I parametri: modificare qui 


In [20]:
ONORARI = 1844.50       # importo degli onorari in euro (numero, punto decimale)
N = 6                   # numero di persone / fatture
CON_IVA = False         # True per includere anche l'IVA nelle verifiche
SPESE_GENERALI = 15     # % spese generali (DM 55/2014, default 15)
CASSA = 4               # % cassa forense (su onorari + spese generali)
IVA = 22                # % IVA (usata solo se CON_IVA = True)
SCARTO_MAX = 40         # massimo scarto (in centesimi) di una quota dalla quota ideale (ONORARI / N)

## Codice: esegui e salta alla soluzione

In [21]:
from decimal import Decimal, ROUND_HALF_UP
from functools import lru_cache
from collections import Counter


def in_centesimi(euro_):
    """1844.50 -> 184450 (intero). Il numero e' letto tramite la sua forma testuale (str), cosi' un float
    non introduce errori binari; vengono considerati al massimo 2 decimali (arrotondamento commerciale)."""
    return int((Decimal(str(euro_)) * 100).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


def mezzo_su(num, den):
    """num/den arrotondato all'intero piu' vicino, meta' verso l'alto (arrotondamento commerciale)."""
    return (2 * num + den) // (2 * den)


@lru_cache(maxsize=None)
def _voci(q0, sg, cassa, iva, con_iva):
    q1 = mezzo_su(sg * q0, 100)
    q2 = mezzo_su(cassa * (q0 + q1), 100)
    q3 = q0 + q1 + q2
    if con_iva:
        i = mezzo_su(iva * q3, 100)
        return (q0, q1, q2, q3, i, q3 + i)
    return (q0, q1, q2, q3)


class fattura:
    """Fattura con le sue voci. Internamente le voci sono in CENTESIMI interi (niente float).
    qta = [onorari, spese generali, cassa, totale] (+ [IVA, totale con IVA]) in euro."""

    SPESE_GENERALI, CASSA, IVA, CON_IVA = 15, 4, 22, False
    NOMI = ["Onorari", "Sp. generali", "Cassa", "Totale", "IVA", "Totale + IVA"]

    def __init__(self, cost=None, voci=None):
        if voci is not None:                       # fattura costruita da voci gia' calcolate (somme, prodotti)
            self.voci = tuple(voci)
        else:                                      # fattura costruita dagli onorari, in euro
            self.voci = self.da_centesimi(in_centesimi(cost)).voci
        self.cost = self.voci[0] / 100

    @classmethod
    def da_centesimi(cls, q0):
        """Fattura con onorari di q0 centesimi."""
        v = _voci(q0, cls.SPESE_GENERALI, cls.CASSA, cls.IVA, cls.CON_IVA)
        f = object.__new__(cls)
        f.voci, f.cost = v, q0 / 100
        return f

    @property
    def qta(self):
        return [v / 100 for v in self.voci]

    def __mul__(self, n):
        return fattura(voci=[v * n for v in self.voci])

    def __rmul__(self, n):
        return self.__mul__(n)

    def __add__(self, other):
        return fattura(voci=[a + b for a, b in zip(self.voci, other.voci)])

    def __eq__(self, other):
        return isinstance(other, fattura) and self.voci == other.voci

    def __hash__(self):
        return hash(self.voci)

    def __repr__(self):
        return f"fattura(qta={self.qta})"


def somma_gruppi(gruppi):
    """gruppi = [(n, fattura), ...]  ->  n1*f1 + n2*f2 + ..."""
    tot = gruppi[0][0] * gruppi[0][1]
    for n, f in gruppi[1:]:
        tot = tot + n * f
    return tot


def scarto_cent(originale, gruppi):
    """Distanza massima (in centesimi) di una quota dalla quota ideale originale/N."""
    Q = originale.voci[0]
    N = sum(n for n, _ in gruppi)
    return max(abs(N * f.voci[0] - Q) for _, f in gruppi) / N


def _gruppi(importi):
    """Lista di importi in centesimi -> [(n, fattura), ...] con importi uguali raggruppati."""
    return [(n, fattura.da_centesimi(c)) for c, n in sorted(Counter(importi).items())]


def cerca_due_gruppi(originale, N, scarto_max):
    """Preferenza 1: quote uguali. Preferenza 2: DUE soli importi (N1 persone a cost1, N2 a cost2),
    scegliendo la coppia piu' vicina alla quota ideale. Restituisce [(n, fattura), ...] oppure None."""
    Q = originale.voci[0]
    base = Q // N
    if Q % N == 0 and N * fattura.da_centesimi(base) == originale:
        return [(N, fattura.da_centesimi(base))]
    migliore = None
    for N1 in range(1, N):
        N2 = N - N1
        for c1 in range(max(1, base - scarto_max - 1), base + scarto_max + 2):
            resto = Q - N1 * c1
            if resto <= 0 or resto % N2:
                continue                              # cost2 deve essere un numero intero di centesimi
            c2 = resto // N2
            if c1 == c2:
                continue
            gruppi = [(N1, fattura.da_centesimi(c1)), (N2, fattura.da_centesimi(c2))]
            s = scarto_cent(originale, gruppi)
            if s > scarto_max or (migliore is not None and s >= migliore[0]):
                continue
            if somma_gruppi(gruppi) == originale:     # N1*split1 + N2*split2 == originale
                migliore = (s, gruppi)
    return migliore[1] if migliore else None


def cerca_liberi(originale, N, scarto_max):
    """Importi liberi (anche tutti diversi), ciascuno in [Q//N - w, Q//N + w] con w minimo.
    Programmazione dinamica sulle somme cumulate delle voci. Restituisce [(n, fattura), ...] o None."""
    Q = originale.voci[0]
    base = Q // N
    idx = (0, 1, 2, 4) if len(originale.voci) == 6 else (0, 1, 2)    # voci indipendenti
    tgt = tuple(originale.voci[i] for i in idx)
    for w in range(scarto_max + 1):
        lo_c, hi_c = max(1, base - w), base + w
        cands = [(c, tuple(fattura.da_centesimi(c).voci[i] for i in idx)) for c in range(lo_c, hi_c + 1)]
        strati = [{tuple(0 for _ in idx)}]
        for k in range(N):
            rimaste = N - k - 1
            lo, hi = Q - rimaste * hi_c, Q - rimaste * lo_c
            nuovi = {ns for st in strati[-1] for c, t in cands
                     for ns in [tuple(a + b for a, b in zip(st, t))] if lo <= ns[0] <= hi}
            if not nuovi:
                break
            strati.append(nuovi)
        else:
            if tgt in strati[N]:
                importi, st = [], tgt
                for k in range(N, 0, -1):
                    for c, t in cands:
                        prima = tuple(a - b for a, b in zip(st, t))
                        if prima in strati[k - 1]:
                            importi.append(c)
                            st = prima
                            break
                return _gruppi(importi)
    return None


def trova_ripartizione(originale, N, scarto_max=SCARTO_MAX):
    """Ordine di preferenza: (1) quote uguali, (2) due soli importi, (3) importi liberi.
    Restituisce [(n, fattura), ...] con  somma_gruppi(...) == originale, oppure None."""
    return cerca_due_gruppi(originale, N, scarto_max) or cerca_liberi(originale, N, scarto_max)


# i parametri scelti sopra diventano le impostazioni della classe
fattura.SPESE_GENERALI, fattura.CASSA, fattura.IVA, fattura.CON_IVA = SPESE_GENERALI, CASSA, IVA, CON_IVA

## Soluzione: visualizza i risultati

In [22]:
originale = fattura(ONORARI)
splitted = fattura(ONORARI / N)                  # quota "ideale" arrotondata, uguale per tutti

is_equal = originale == (splitted * N)

print(f"Originale qta: {originale.qta}")
print(f"Splitted * N : {(splitted * N).qta}")
print(f"Sono uguali? {is_equal}\n")

Originale qta: [1844.5, 276.68, 84.85, 2206.03]
Splitted * N : [1844.52, 276.66, 84.84, 2206.02]
Sono uguali? False



In [23]:
if not is_equal:
    gruppi = trova_ripartizione(originale, N)
    if gruppi is None:
        print(f"Nessuna ripartizione trovata entro {SCARTO_MAX} centesimi dalla quota ideale: aumenta SCARTO_MAX.")
    else:
        tipo = {1: "quote tutte uguali", 2: "due soli importi"}.get(len(gruppi), "importi liberi")
        print(f"--- Ripartizione trovata: {tipo} ---")
        for i, (n, split) in enumerate(gruppi, 1):
            print(f"N{i} = {n}, cost{i} = {split.cost:.2f}")
            print(f"split{i} qta: {split.qta}")
        tot = somma_gruppi(gruppi)
        print(f"Scarto massimo dalla quota ideale: {scarto_cent(originale, gruppi):.2f} centesimi")
        print("Somma dei gruppi == originale:", tot == originale)
        assert tot == originale

--- Ripartizione trovata: due soli importi ---
N1 = 1, cost1 = 307.50
split1 qta: [307.5, 46.13, 14.15, 367.78]
N2 = 5, cost2 = 307.40
split2 qta: [307.4, 46.11, 14.14, 367.65]
Scarto massimo dalla quota ideale: 8.33 centesimi
Somma dei gruppi == originale: True


## Note sui Limiti:

Nessuna garanzia teorica che esista sempre una soluzione; se non c'è, il notebook lo dice. Pensato per N piccoli o medi (decine di quote al massimo).